# 🔁 01 · 动态规划：策略评估 / 策略迭代 / 价值迭代

> 面试定位：DP 是**有模型（model-based）**的规划方法——已知 $P$ 和 $R$，不与环境交互，直接解贝尔曼方程。
> 本篇沿用 4×4 网格世界，手写三种算法并对比收敛速度。
> 标记：🔴 必背 / 🟡 掌握。

---

## 1. 动态规划三件套（🔴 总览）

| 算法 | 解决的问题 | 核心更新式 | 是否用 max |
|---|---|---|---|
| **策略评估** | 给定 π，求 V^π | $V_{k+1} = \sum_a \pi(a|s)\sum_{s'} P[\cdot]$ | 否（按策略平均） |
| **策略迭代** | 求最优 π | 评估 V^π → **贪心改进** π' = argmax | 改进时用 |
| **价值迭代** | 求最优 V* | $V_{k+1} = \max_a \sum_{s'} P[\cdot]$ | 是（每步 max） |

### 使用前提（背）

- **知道完整模型**：转移概率 $P(s'|s,a)$ 与奖励 $R(s,a,s')$
- **有限状态/动作**：表格可存下（大规模 → 函数近似，见 DQN 篇）
- **环境静态**：P 和 R 不随时间变（非平稳环境 DP 失效）

### 为什么"动态规划"叫规划而不是学习

RL 两大分支：**规划（planning，有模型，DP/树搜索）** vs **学习（learning，无模型，MC/TD）**。DP 不产生新经验，只在已有模型上计算——所以是"规划"。

---


## 2. 策略评估：迭代求解 V^π（🔴）

### 更新式（同步迭代）

$$V_{k+1}(s) = \sum_{a} \pi(a|s) \sum_{s'} P(s'|s,a)\left[ R(s,a,s') + \gamma V_k(s') \right]$$

### 关键点

- **同步更新**：用旧 $V_k$ 算所有 $V_{k+1}$，再整体替换（异步会破坏收敛保证）
- **收敛判据**：$\max_s |V_{k+1}(s) - V_k(s)| < \varepsilon$
- **复杂度**：每轮 O(|S|·|A|·|S'|)，|S'| 是每个 (s,a) 的可能后继数

### 与前面 00 篇的区别

00 篇已经演示过随机策略评估；本篇把它做成**可复用函数**，并为策略迭代服务。

---


In [ ]:
# policy_eval.py —— 通用策略评估函数（同步迭代 + 收敛检查）
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    """确定性网格转移: [(概率, s', r)]"""
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

def policy_eval(pi, tol=1e-8, max_iter=500):
    """pi: (N, 4) 概率矩阵"""
    V = np.zeros(N)
    for it in range(max_iter):
        V_new = np.zeros(N)
        for s in range(N):
            v = 0.0
            for a in range(4):
                for p, s2, r in trans(s, a):
                    v += pi[s, a] * p * (r + GAMMA * V[s2])
            V_new[s] = v
        if np.max(np.abs(V_new - V)) < tol:
            return V_new, it + 1
        V = V_new
    return V, max_iter

pi_rand = np.full((N, 4), 0.25)
V, iters = policy_eval(pi_rand)
print(f'随机策略评估收敛: {iters} 轮')
print(np.round(V.reshape(SIZE, SIZE), 2))

# 更"聪明"的启发式策略: 往终点方向走的概率更大
def heuristic_pi():
    pi = np.zeros((N, 4))
    for s in range(N):
        r, c = divmod(s, SIZE)
        if s == N - 1:
            pi[s, 0] = 1.0
            continue
        # 朝终点方向的动作给更大概率
        best = []
        if r < 3: best.append(1)   # 下
        if c < 3: best.append(3)   # 右
        if r > 0: best.append(0)   # 上
        if c > 0: best.append(2)   # 左
        for a in best:
            pi[s, a] = 0.7 / len(best)
        pi[s] += 0.3 / 4 * (pi[s] == 0).astype(float)
        pi[s] /= pi[s].sum()
    return pi

V2, iters2 = policy_eval(heuristic_pi())
print(f'启发式策略评估收敛: {iters2} 轮, 起点价值 {V2[0]:.3f} (随机策略 {V[0]:.3f})')
print('→ 策略越好, 同样迭代下起点价值越高')


## 3. 策略改进定理（🔴）

### 贪心改进

给定 $V^{\pi}$，构造新策略：

$$\pi'(s) = \arg\max_{a} Q^{\pi}(s,a) = \arg\max_{a} \sum_{s'} P(s'|s,a)\left[ R + \gamma V^{\pi}(s') \right]$$

### 策略改进定理

若 $Q^{\pi}(s, \pi'(s)) \ge V^{\pi}(s)$ 对所有 $s$ 成立，则 $\pi'$ **不差于** $\pi$，且至少在一个状态上严格更好时，$\pi'$ 严格更优。

### 直觉（背）

**只要"在某个状态换成更优动作"至少不亏，整个策略就变好**——因为未来同样按 π 走（单步偏离原则）。

### 终止条件

改进后策略不再变化（$\pi_{k+1}=\pi_k$）→ 此时 $V^{\pi_k} = V^*$，达到最优。

---


In [ ]:
# policy_iteration.py —— 策略迭代: 评估 → 贪心改进 → 直到稳定
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

def policy_eval(pi, tol=1e-8, max_iter=500):
    V = np.zeros(N)
    for it in range(max_iter):
        V_new = np.zeros(N)
        for s in range(N):
            v = 0.0
            for a in range(4):
                for p, s2, r in trans(s, a):
                    v += pi[s, a] * p * (r + GAMMA * V[s2])
            V_new[s] = v
        if np.max(np.abs(V_new - V)) < tol:
            return V_new
        V = V_new
    return V

def greedy_improve(V):
    """由 V 构造确定性贪心策略"""
    pi = np.zeros((N, 4))
    for s in range(N):
        q = [sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a)) for a in range(4)]
        pi[s, int(np.argmax(q))] = 1.0
    return pi

# 从随机策略出发
pi = np.full((N, 4), 0.25)
for k in range(50):
    V = policy_eval(pi)
    pi_new = greedy_improve(V)
    if np.array_equal(pi, pi_new):
        print(f'策略迭代第 {k} 轮后策略稳定')
        break
    pi = pi_new

print('最优策略（每格箭头, * 终点）:')
ACTIONS = ['↑', '↓', '←', '→']
for r in range(SIZE):
    row = ''
    for c in range(SIZE):
        s = r * SIZE + c
        row += ('  *' if s == N - 1 else '  ' + ACTIONS[int(np.argmax(pi[s]))])
    print(row)
print()
print('起点价值 V*(0,0) =', round(V[0], 3))
print('对比 00 篇值迭代结果应一致 → 两种算法殊途同归')


## 4. 价值迭代（🔴 回顾 + 对比）

### 更新式

$$V_{k+1}(s) = \max_{a} \sum_{s'} P(s'|s,a)\left[ R(s,a,s') + \gamma V_k(s') \right]$$

### 价值迭代 vs 策略迭代（面试对比表，背）

| 维度 | 策略迭代 | 价值迭代 |
|---|---|---|
| 每轮做 | 完整评估 V^π（内部多次迭代）+ 改进 | 一步 max 更新（不评估任何策略） |
| 每轮成本 | 高（评估子循环） | 低（单次扫描） |
| 收敛轮数 | 少（一般几轮） | 多（每轮只推进一点） |
| 什么时候好 | 状态少 | 状态多、评估贵 |
| 中间量 | 有明确策略序列 π0→π1→… | 只有价值序列 V0→V1→… |

### 广义策略迭代（GPI，🔴 概念）

**评估与改进交错进行**：策略迭代是"评估收敛→改进"，价值迭代是"评估一步→改进一步"。**几乎一切 RL 方法（包括 DQN/PPO）都是 GPI 的实例**——这是面试超高频的一句话。

> **面试金句**：DQN 里"用回放数据更新 Q"是评估、"按 ε-greedy 取动作"是改进，两者交替 → GPI。答出这句能串联整条 RL 知识线。

---


In [ ]:
# convergence_compare.py —— 收敛实验: 策略迭代 vs 价值迭代 轮数/成本
import time
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

def policy_iteration(verbose=True):
    pi = np.full((N, 4), 0.25)
    for k in range(50):
        # 评估
        V = np.zeros(N)
        for _ in range(500):
            V_new = np.zeros(N)
            for s in range(N):
                v = 0.0
                for a in range(4):
                    for p, s2, r in trans(s, a):
                        v += pi[s, a] * p * (r + GAMMA * V[s2])
                V_new[s] = v
            if np.max(np.abs(V_new - V)) < 1e-8:
                V = V_new
                break
            V = V_new
        # 改进
        pi_new = np.zeros((N, 4))
        for s in range(N):
            q = [sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a)) for a in range(4)]
            pi_new[s, int(np.argmax(q))] = 1.0
        if np.array_equal(pi, pi_new):
            return k + 1, V[0]
        pi = pi_new
    return 50, V[0]

def value_iteration():
    V = np.zeros(N)
    for k in range(2000):
        V_new = np.zeros(N)
        for s in range(N):
            best = max(sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a)) for a in range(4))
            V_new[s] = best
        if np.max(np.abs(V_new - V)) < 1e-8:
            return k + 1, V_new[0]
        V = V_new
    return 2000, V[0]

t0 = time.time(); pk, pv = policy_iteration(); t1 = time.time()
vk, vv = value_iteration(); t2 = time.time()
print(f'策略迭代: {pk} 轮, 起点价值 {pv:.3f}, 耗时 {t1-t0:.3f}s')
print(f'价值迭代: {vk} 轮, 起点价值 {vv:.3f}, 耗时 {t2-t1:.3f}s')
print('→ 小网格下两种算法结果一致; 策略迭代轮数少但每轮贵(完整评估)')


## 5. 异步 DP 与实时 DP（🟢 了解）

### 为什么提异步

同步 DP 每轮扫**全部状态**，状态多时浪费——很多状态的价值早就不变了。

### 异步策略评估（Asynchronous DP）

- 每轮只更新**部分状态**（如按优先级：价值变化大的状态先更新）
- 保证收敛（只要所有状态被无限次更新）；典型实现：**优先扫描（Prioritized Sweeping）**

### 实时 DP（Real-time DP）

- 只更新**智能体实际访问到的状态**——与环境交互驱动更新
- 是 DP 与学习的桥梁：**模型可以来自学习**（learned model + planning = **Model-Based RL**，如 Dyna-Q）

> **面试点**：Dyna-Q 同时做"学习（TD 更新真实经验）"和"规划（用学习到的模型做模拟经验更新）"，是融合 planning 与 learning 的代表——见 03 篇补充。

---

## 6. 自测清单

- [ ] L1：默写策略评估、策略改进、价值迭代三个更新式
- [ ] L1：说清策略迭代 vs 价值迭代的每轮成本与收敛轮数差异
- [ ] L1：讲出"策略改进定理"的单步偏离直觉
- [ ] L2：解释为什么 DQN/PPO 都符合 GPI 框架
- [ ] L2：手写 4×4 网格的策略迭代（20 分钟内, 允许查公式）
- [ ] L2：能说出异步 DP 与同步 DP 的区别和收敛条件

>  下一篇 `02-蒙特卡洛与时序差分`：**无模型** 学习——不已知 P/R，靠采样估计。


## 7. 策略评估的演化过程：价值如何"从 0 长出来"（🟡 直觉）

### 每轮更新在做什么

$V_0=0$ → 第一轮算出"走一步能拿多少" → 第二轮加上"下一步价值的期望" → ……每轮把**更远一层的未来**考虑进来：

```text
V1(s) = 只算1步:  撞墙-1 / 走到终点附近 +10
V2(s) = 1步 + 折扣×(1步): 看到2步
Vk(s) = 看到 k 步 → k→∞ 收敛到 V^π
```

### 为什么能收敛

$V_{k+1} = R + \gamma P V_k$ 中 $\gamma P$ 的谱半径 $<1$ → 每次迭代误差乘 $\gamma$ → **指数收敛**（压缩映射）。

### 观察要点

打印每 5 轮的价值快照：前期变化大（远处价值"传导"过来），后期只微调。


In [ ]:
# policy_eval_frames.py —— 打印策略评估的演化帧（每 5 轮）
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

pi = np.full((N, 4), 0.25)
V = np.zeros(N)
for it in range(1, 61):
    Vn = np.zeros(N)
    for s in range(N):
        Vn[s] = sum(pi[s, a] * p * (r + GAMMA * V[s2])
                    for a in range(4) for p, s2, r in trans(s, a))
    V = Vn
    if it % 10 == 0:
        print(f'--- 第 {it:2d} 轮 (起点价值 {V[0]:6.3f}) ---')
        print(np.round(V.reshape(SIZE, SIZE), 2))

print()
print('观察: 价值从 0 起步, 前 10 轮大步逼近, 之后只微调(误差每轮×γ)')


## 8. 策略改进的单步效果（🟡）

### 演示设计

从**随机策略**开始：

1. 评估随机策略得 $V^{\pi_{\text{rand}}}$
2. 对每个状态贪心改进一次 → $\pi'$
3. 展示 $\pi'$ 与随机策略的差异——**改进定理说它至少不差**

### 关键观察

一次改进后策略已经"像样"（指向终点方向），但还不是最优——因为 $V^{\pi_{\text{rand}}}$ 是随机策略的价值，不是最优价值。**策略迭代就是"评估→改进"反复**（01 篇 3 节已实现）。

---


In [ ]:
# greedy_one_step.py —— 从随机策略做一次贪心改进, 看策略变"聪明"多少
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

pi_rand = np.full((N, 4), 0.25)
# 评估随机策略
V = np.zeros(N)
for _ in range(500):
    Vn = np.zeros(N)
    for s in range(N):
        Vn[s] = sum(pi_rand[s, a] * p * (r + GAMMA * V[s2])
                    for a in range(4) for p, s2, r in trans(s, a))
    V = Vn

# 一次贪心改进
pi_new = np.zeros((N, 4))
for s in range(N):
    q = [sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a)) for a in range(4)]
    pi_new[s, int(np.argmax(q))] = 1.0

# 随机策略 vs 改进后策略的起点价值
def eval_policy(pi):
    Vp = np.zeros(N)
    for _ in range(500):
        Vn = np.zeros(N)
        for s in range(N):
            Vn[s] = sum(pi[s, a] * p * (r + GAMMA * Vp[s2])
                        for a in range(4) for p, s2, r in trans(s, a))
        Vp = Vn
    return Vp[0]

print(f'随机策略  起点价值: {eval_policy(pi_rand):.3f}')
print(f'一次改进后起点价值: {eval_policy(pi_new):.3f}')
print('→ 满足改进定理: 单步贪心改进后价值提升(不下降)')


## 9. 两种算法的收敛过程对比（🟡 数值实验）

### 实验

- 策略迭代：记录每轮"改进后的起点价值"
- 价值迭代：记录每轮"V 的起点价值"
- 对比两者的收敛速度曲线

### 预期结论

- 策略迭代轮数少但每轮贵（内部完整评估）
- 价值迭代轮数多但每轮便宜（一步 max）
- **两者最终收敛到同一个 $V^*$**

---


In [ ]:
# converge_trace.py —— 策略迭代 vs 价值迭代: 起点价值随"外部轮数"的变化
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

def full_eval(pi):
    V = np.zeros(N)
    for _ in range(500):
        Vn = np.zeros(N)
        for s in range(N):
            Vn[s] = sum(pi[s, a] * p * (r + GAMMA * V[s2])
                        for a in range(4) for p, s2, r in trans(s, a))
        V = Vn
    return V

# 策略迭代轨迹
pi = np.full((N, 4), 0.25)
pi_trace = []
for _ in range(15):
    V = full_eval(pi)
    pi_trace.append(V[0])
    pi_new = np.zeros((N, 4))
    for s in range(N):
        q = [sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a)) for a in range(4)]
        pi_new[s, int(np.argmax(q))] = 1.0
    if np.array_equal(pi, pi_new):
        break
    pi = pi_new

# 价值迭代轨迹
V = np.zeros(N)
vi_trace = []
for _ in range(30):
    Vn = np.zeros(N)
    for s in range(N):
        Vn[s] = max(sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a)) for a in range(4))
    V = Vn
    vi_trace.append(V[0])

print(f'策略迭代: {len(pi_trace)} 轮收敛, 起点价值轨迹:')
print('  ', [round(v, 3) for v in pi_trace])
print(f'价值迭代: 前 8 轮起点价值轨迹:')
print('  ', [round(v, 3) for v in vi_trace[:8]])
print()
print(f'最终一致: 策略迭代 {pi_trace[-1]:.3f} vs 价值迭代 {vi_trace[-1]:.3f}')


## 10. 随机 MDP 生成器：两种算法在"没见过"的环境上验证（🟡 泛化测试）

### 为什么做这个实验

前面都是同一个 4×4 网格——万一算法只对"这个环境"有效呢？**随机生成大量小 MDP**（随机 P/R），两种算法都应收敛到同一最优值，且策略一致。

### 随机 MDP 生成

- 状态 6 个、动作 2 个
- 每个 (s,a) 随机 2~3 个后继、随机概率、随机奖励 ∈ [0,1]
- γ=0.9

> **验收标准**：100 个随机 MDP，两种算法 100% 收敛且最优价值一致（差值 < 1e-6）。


In [ ]:
# random_mdp.py —— 100 个随机 MDP 上验证: 策略迭代 == 价值迭代
import numpy as np

def make_random_mdp(ns=6, na=2, gamma=0.9, seed=0):
    rng = np.random.RandomState(seed)
    P = {}      # (s,a) -> [(p, s2, r)]
    for s in range(ns):
        for a in range(na):
            k = rng.randint(2, 4)                      # 2~3 个后继
            succ = rng.choice(ns, size=k, replace=False)
            probs = rng.dirichlet(np.ones(k))
            rew = rng.rand(k)
            P[(s, a)] = list(zip(probs, succ, rew))
    return P, gamma

def value_iteration(P, ns, na, gamma, tol=1e-9):
    V = np.zeros(ns)
    for _ in range(2000):
        Vn = np.zeros(ns)
        for s in range(ns):
            Vn[s] = max(sum(p * (r + gamma * V[s2]) for p, s2, r in P[(s, a)]) for a in range(na))
        if np.max(np.abs(Vn - V)) < tol:
            V = Vn
            break
        V = Vn
    return V

def policy_iteration(P, ns, na, gamma):
    pi = {s: 0 for s in range(ns)}
    for _ in range(100):
        V = np.zeros(ns)
        for _ in range(2000):
            Vn = np.zeros(ns)
            for s in range(ns):
                a = pi[s]
                Vn[s] = sum(p * (r + gamma * V[s2]) for p, s2, r in P[(s, a)])
            if np.max(np.abs(Vn - V)) < 1e-9:
                V = Vn
                break
            V = Vn
        improved = False
        for s in range(ns):
            q = [sum(p * (r + gamma * V[s2]) for p, s2, r in P[(s, a)]) for a in range(na)]
            a_best = int(np.argmax(q))
            if pi[s] != a_best:
                pi[s] = a_best
                improved = True
        if not improved:
            break
    # 由 π 算价值
    V = np.zeros(ns)
    for _ in range(2000):
        Vn = np.zeros(ns)
        for s in range(ns):
            a = pi[s]
            Vn[s] = sum(p * (r + gamma * V[s2]) for p, s2, r in P[(s, a)])
        if np.max(np.abs(Vn - V)) < 1e-9:
            V = Vn
            break
        V = Vn
    return V

agree = 0
worst = 0.0
for seed in range(100):
    P, gamma = make_random_mdp(seed=seed)
    V_vi = value_iteration(P, 6, 2, gamma)
    V_pi = policy_iteration(P, 6, 2, gamma)
    diff = np.max(np.abs(V_vi - V_pi))
    agree += diff < 1e-6
    worst = max(worst, diff)

print(f'100 个随机 MDP: 两种算法最优价值一致 {agree}/100')
print(f'最大差异: {worst:.2e}')
print('→ 数值上验证: 策略迭代与价值迭代求解的是同一个贝尔曼最优方程')


## 11. 同步 vs 异步（in-place）更新（🟡 工程细节）

### 两种更新方式

| | 同步（Jacobi） | 异步/in-place（Gauss-Seidel） |
|---|---|---|
| 用旧值还是新值 | 全部用上一轮的 $V_k$ | 用**已更新的**新值 |
| 收敛速度 | 慢（信息传播一轮只用旧值） | 快（同轮内新值立即参与） |
| 实现 | 需要临时数组 V_new | 原地改一个数组 |
| 收敛保证 | 必收敛 | 也收敛（有界步长/满足条件） |

### 直觉

in-place 更新 = 状态按顺序刷，后面的状态立即看到前面的新值 → 信息传播更快。经典结论：**Gauss-Seidel 通常比 Jacobi 快 ~2 倍**。

---


In [ ]:
# sync_vs_async.py —— 同步 vs in-place 收敛速度对比
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

pi = np.full((N, 4), 0.25)

def solve_sync(tol=1e-8):
    V = np.zeros(N)
    for it in range(2000):
        Vn = np.zeros(N)
        for s in range(N):
            Vn[s] = sum(pi[s, a] * p * (r + GAMMA * V[s2])
                        for a in range(4) for p, s2, r in trans(s, a))
        if np.max(np.abs(Vn - V)) < tol:
            return it + 1
        V = Vn
    return 2000

def solve_async(tol=1e-8):
    V = np.zeros(N)
    for it in range(2000):
        diff = 0.0
        for s in range(N):
            new = sum(pi[s, a] * p * (r + GAMMA * V[s2])
                      for a in range(4) for p, s2, r in trans(s, a))
            diff = max(diff, abs(new - V[s]))
            V[s] = new                       # 立即写入, 后续状态用新值
        if diff < tol:
            return it + 1
    return 2000

print(f'同步更新收敛轮数: {solve_sync()}')
print(f'in-place 更新收敛轮数: {solve_async()}')
print('→ in-place 通常显著更快(信息同轮内传播)')


## 12. 广义策略迭代 GPI：一切 RL 的统一框架（🔴 必背升华）

### 一句话

**"评估价值 + 改进策略"两个过程交替进行，就是 GPI。**

- 策略迭代：评估到收敛 → 改进（极端的交替）
- 价值迭代：评估一步 → 改进一步（另一个极端）
- DQN/PPO：评估（用数据更新 Q/价值）与改进（按价值取动作/更新策略）交错 → **也是 GPI**

### 为什么 GPI 重要（面试）

> 面试官问"RL 算法的统一框架是什么"，答案是 **GPI**：所有 RL 方法都在做"价值估计"与"策略改善"两个循环。回答时举例 DQN（回放更新 = 评估，ε-greedy = 改进）直接封神。

### 两个圈互相"促进但不等待"

评估不完全也能改进，改进不完全也能继续评估——只要两个过程**都朝着正确方向**，整体收敛。这是 GPI 收敛性的核心直觉。

---


In [ ]:
# gpi_demo.py —— 交错式 GPI: 每轮只评估 1 步就改进, 也能收敛
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

# GPI: 评估1步(不收敛) -> 改进1次 -> 重复
pi = np.full((N, 4), 0.25)
V = np.zeros(N)
for k in range(100):
    # 评估 1 步（不等到收敛）
    Vn = np.zeros(N)
    for s in range(N):
        Vn[s] = sum(pi[s, a] * p * (r + GAMMA * V[s2])
                    for a in range(4) for p, s2, r in trans(s, a))
    V = Vn
    # 改进 1 次
    pi_new = np.zeros((N, 4))
    for s in range(N):
        q = [sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a)) for a in range(4)]
        pi_new[s, int(np.argmax(q))] = 1.0
    if np.array_equal(pi, pi_new):
        print(f'GPI 第 {k+1} 轮策略稳定')
        break
    pi = pi_new

print('交错式 GPI 学到的策略（箭头, * 终点）:')
ACTIONS = ['↑', '↓', '←', '→']
for r in range(SIZE):
    print(''.join('  *' if r * SIZE + c == N - 1 else '  ' + ACTIONS[int(np.argmax(pi[r * SIZE + c]))]
                  for c in range(SIZE)))
print()
print('结论: 评估不需要收敛, 交错推进照样收敛到最优 —— GPI 的威力')


## 13. 规模压力测试：8×8 与 20×20 网格（🟡 性能直觉）

### 实验目的

观察 DP 随状态数增长的**轮数与耗时**：表格法在 |S|=64/400 时依然秒级——真正的瓶颈是 |S| 更大时的**内存与单轮成本**，以及真实环境中 P/R 未知。

### 意义（面试）

> "DP 能解多大问题？" → 状态数千万级以内的表格可以；真实大规模靠函数近似（神经网络），因为 DP 需要完整的 P 表（内存 O(|S|²|A|)）与每轮全扫。

---


In [ ]:
# scale_test.py —— 8x8 与 20x20 网格的值迭代: 轮数/耗时
import time
import numpy as np

def grid_vi(size, gamma=0.9):
    N = size * size
    MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    def trans(s, a):
        if s == N - 1:
            return [(1.0, s, 0.0)]
        r, c = divmod(s, size)
        dr, dc = MOVE[a]
        nr, nc = r + dr, c + dc
        if not (0 <= nr < size and 0 <= nc < size):
            return [(1.0, s, -1.0)]
        s2 = nr * size + nc
        return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]
    V = np.zeros(N)
    for it in range(5000):
        Vn = np.zeros(N)
        for s in range(N):
            Vn[s] = max(sum(p * (r + gamma * V[s2]) for p, s2, r in trans(s, a)) for a in range(4))
        if np.max(np.abs(Vn - V)) < 1e-8:
            return it + 1, V[0]
        V = Vn
    return 5000, V[0]

for size in [4, 8, 20]:
    t0 = time.time()
    it, v0 = grid_vi(size)
    print(f'{size}x{size} 网格: 状态数 {size*size}, 值迭代 {it} 轮收敛, 起点价值 {v0:.3f}, 耗时 {time.time()-t0:.2f}s')

print()
print('规模感: 400 状态仍秒级; 但真实环境的 P 表(400x4x400)已是 64 万项')


## 14. 截断评估（🟢 实际工程的近似）

### 问题

策略迭代里每次"完整评估到收敛"很贵——**评估不收敛也行**（GPI 理论保证）。

### 截断评估做法

- 每轮只评估 $k$ 步（如 1~3 步）就改进
- 收益：每轮便宜、总轮数可能增加但整体更快
- 极端就是价值迭代（评估 1 步）

### 数值验证

下面比较"完整评估"与"3 步截断评估"的总耗时。

---


In [ ]:
# truncated_eval.py —— 完整评估 vs 截断评估(每轮3步) 策略迭代耗时对比
import time
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

def policy_iter(k_eval, max_rounds=100):
    pi = np.full((N, 4), 0.25)
    V = np.zeros(N)
    for _ in range(max_rounds):
        for _ in range(k_eval):
            Vn = np.zeros(N)
            for s in range(N):
                Vn[s] = sum(pi[s, a] * p * (r + GAMMA * V[s2])
                            for a in range(4) for p, s2, r in trans(s, a))
            V = Vn
        pi_new = np.zeros((N, 4))
        for s in range(N):
            q = [sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a)) for a in range(4)]
            pi_new[s, int(np.argmax(q))] = 1.0
        if np.array_equal(pi, pi_new):
            return V[0]
        pi = pi_new
    return V[0]

for k in [1, 3, 10, 9999]:
    t0 = time.time()
    v0 = policy_iter(k if k < 9999 else 200)
    dt = time.time() - t0
    tag = '完整评估(200步/轮)' if k == 9999 else f'每轮评估{k}步'
    print(f'{tag:<18} 起点价值 {v0:.3f}  耗时 {dt:.3f}s')
print()
print('结论: 截断评估总耗时可能更短(评估不收敛也够用) —— GPI 思想的工程化')


## 15. 面试追问速答扩展（🔴）

1. **Q: 策略迭代为什么一定收敛？** A: 每轮改进价值单调不减（改进定理）+ 策略数有限 → 有限步内到最优。
2. **Q: 值迭代的复杂度？** A: 每轮 O(|S|²|A|)；收敛轮数与 γ 有关（误差 ≤ γ^k 界）。
3. **Q: DP 需要什么前提？** A: 完整模型 P/R + 表格状态 + 静态环境。
4. **Q: 环境模型从哪来？** A: 物理仿真/规则引擎直接给；或从数据学（Model-Based RL / Dyna）。
5. **Q: 为什么真实 RL 很少直接 DP？** A: 状态巨大、P/R 未知、环境非平稳——DP 是"规划"，学习是"数据驱动"。
6. **Q: 异步 DP 的收敛条件？** A: 所有状态被无限次更新（保证性条件），优先级可加速。
7. **Q: 优先扫描（Prioritized Sweeping）？** A: 按 TD 误差大小排队更新——价值变化大的状态先算。
8. **Q: DP 与最优控制的联系？** A: 贝尔曼方程 = 离散时间的**动态规划**（Bellman 1957），最优控制是其连续版（HJB）。

---

## 16. 自测清单（01 篇完整版）

- [ ] L1：默写策略评估/改进/价值迭代三个更新式
- [ ] L1：说清 GPI 是什么、为什么 DQN/PPO 都是 GPI
- [ ] L1：解释同步 vs in-place 更新的收敛速度差异
- [ ] L2：手写随机 MDP 生成器并跑通两种算法一致性
- [ ] L2：理解"评估 = 解线性方程组 (I-γP)V=R"
- [ ] L3：讲清 DP 的规模瓶颈与函数近似的动机
- [ ] L3：能举出截断评估/优先扫描等工程近似

>  下一篇 `02-蒙特卡洛与时序差分`：**无模型**学习——脱离 P/R 的采样估计。


## 17. 收敛性数值实验：收缩系数 γ 的影响（🟡）

### 理论

$|V_{k+1} - V^*|_\infty \le \gamma^k |V_0 - V^*|_\infty$ —— 每轮误差至多乘 γ。

### 实验

扫 γ ∈ {0.5, 0.8, 0.95}，记录值迭代**达到 ε 精度**的轮数。

---


In [ ]:
# gamma_conv.py —— γ 收敛速度: 达到 1e-6 精度的轮数
import numpy as np

def value_iter_iters(gamma, size=4, tol=1e-6):
    N = size * size
    MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    def trans(s, a):
        if s == N - 1:
            return [(1.0, s, 0.0)]
        r, c = divmod(s, size)
        dr, dc = MOVE[a]
        nr, nc = r + dr, c + dc
        if not (0 <= nr < size and 0 <= nc < size):
            return [(1.0, s, -1.0)]
        s2 = nr * size + nc
        return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]
    V = np.zeros(N)
    for it in range(10000):
        Vn = np.zeros(N)
        for s in range(N):
            Vn[s] = max(sum(p * (r + gamma * V[s2]) for p, s2, r in trans(s, a)) for a in range(4))
        if np.max(np.abs(Vn - V)) < tol:
            return it + 1
        V = Vn
    return 10000

print(f'{"γ":<8}{"收敛轮数(tol=1e-6)":>20}')
for gamma in [0.5, 0.8, 0.9, 0.95]:
    print(f'{gamma:<8}{value_iter_iters(gamma):>20}')
print()
print('→ γ→1 时收敛急剧变慢(误差每轮只乘 0.95); γ=1 且无限 horizon 可能不收敛')


## 18. 精确解与不动点：多余的线性代数视角（🟢 面试加分）

### 价值迭代的一个理解

$V_{k+1} = T V_k$ 是不动点迭代，等价于求解不动点 $V = TV$。在表格式、有限维下：

- 策略评估：线性方程 $(I - \gamma P)V = R$
- 价值迭代：非线性$($因为max$)$，但仍是压缩映射

### 面试加分句

> "DP 是**离线规划**：先有模型再算；如果模型是学出来的，就变成 **Model-Based RL**——用学到的转移做'想象中的策略迭代'。"

---


## 19. 面试场景：手写 3×3 网格值迭代（🔴 临场模板）

### 手写步骤（背）

```text
① 定义: N=9, 动作4, MOVE, 终点8
② trans(s,a): 返回后继续列表 [(p,s',r)]
③ 迭代: Vn[s]=max_a Σ p(r+γV[s2]); 收敛判 max|Δ|<1e-9
④ 策略: pi[s]=argmax_a 同样表达式
⑤ 打印: 网格价值 + 箭头
```

### 常见坑

- 忘了终点是**吸收态**（self-loop）
- 撞墙处理：留在原地 vs 原地+惩罚——要与面试官对齐假设再写
- max 里没写全四个动作

下面给一个可直接背诵的 3×3 完整实现。

---


In [ ]:
# grid33_handscript.py —— 3×3 值迭代标准答案(背诵用)
import numpy as np

SIZE, GAMMA = 3, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_b(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_b(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

V = np.zeros(N)
for _ in range(1000):
    Vn = np.zeros(N)
    for s in range(N):
        Vn[s] = max(sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a)) for a in range(4))
    if np.max(np.abs(Vn - V)) < 1e-9:
        V = Vn
        break
    V = Vn

print('3×3 值迭代 V*:')
print(np.round(V.reshape(SIZE, SIZE), 2))
policy = []
for s in range(N):
    q = [sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a)) for a in range(4)]
    policy.append(['↑', '↓', '←', '→'][int(np.argmax(q))])
print('最优策略:')
for r in range(SIZE):
    print(' '.join('*' if r * SIZE + c == N - 1 else policy[r * SIZE + c] for c in range(SIZE)))


## 20. 面试追问速答终版（🔴 01 篇收官）

1. **Q: DP 是学习还是规划？** A: 规划（planning）——用已知模型计算，不与环境交互采样。
2. **Q: 策略评估与值迭代的更新式写出来？** A: 一个用 Σπ 一个用 max；其余结构相同。
3. **Q: GPI 与"近似"的关系？** A: 评估/改进都允许不完美，交替即可收敛——一切 RL 的范式。
4. **Q: 值迭代每步都贪心，为什么还要多轮？** A: V 未收敛前贪心只是"局部最优"；需要多轮信息传播。
5. **Q: in-place 更新是否正确？** A: 正确（偏 Gauss-Seidel），且更快；但严格性要满足顺序遍历。
6. **Q: 随机 MDP 下 DP 仍可行？** A: 是——DP 只要求 P/R 已知，不要求确定性。
7. **Q: learn model 与 learn policy 的区别？** A: 学模型=学环境动力学（MBRL）；学策略=直接学决策（policy-based）。
8. **Q: 值迭代与策略迭代在小问题上怎么选？** A: 策略迭代轮数少但每轮贵；小问题两者都几乎瞬间，讲清差异即可。

---

## 21. 自测清单（01 篇终版）

- [ ] 旧条目全部保持可背写
- [ ] 新增：能解释 γ 对收敛轮数的影响（0.5 vs 0.95）
- [ ] 新增：10 分钟内手写 3×3 值迭代（含吸收态与撞墙）
- [ ] 新增：说清"学模型则变 MBRL"的一句话


## 22. DP 三算法对照总表（🔴 考前速背）

| | 策略评估 | 策略迭代 | 价值迭代 |
|---|---|---|---|
| 目标 | V^π | π* | V* |
| 更新 | $\Sigma_\pi$ | 评估+贪心改进 | $\max$ |
| 每轮成本 | O(\|S\|²\|A\|) | 评估贵+改进 | 便宜 |
| 轮数 | 数百 | 少 | 多 |
| 用模型 | 是 | 是 | 是 |
| 本质 | 线性方程 | 交替优化 | 压缩迭代 |

### 关系图

```text
策略评估 →(贪心改进)→ 策略迭代
价值迭代 = 评估1步+改进1步 反复(极端 GPI)
一切 RL = GPI(评估+改进) 的实例
```


## 23. 值迭代 vs 策略迭代的直观类比（🟢 记忆锚点）

```text
策略迭代 = "彻底想清楚再动手"：每轮把当前路线的价值算到收敛，再换路线
价值迭代 = "边想边动手"：每轮只前看一步就调整方向，反复多轮
GPI     = "边想边动手"的通用版（评估/改进都可截断）
```

> 面试被问"两算法谁快"→ 答：**没有绝对**——状态少策略迭代轮数少；状态多价值迭代每轮便宜。真正工业界用 GPI 思想的近似版（DQN/PPO）。


## 24. 附录：常用符号速查（🔴 写题不再卡壳）

| 符号 | 含义 |
|---|---|
| $S_t, A_t, R_t$ | 时刻 t 的状态/动作/奖励 |
| $\pi(a\mid s)$ | 策略（条件概率） |
| $d^\pi(s)$ | 策略 π 下的稳态分布 |
| $V^\pi, Q^\pi$ | 策略 π 的价值 |
| $V^*, Q^*$ | 最优价值 |
| $\delta_t$ | TD 误差 |
| $\alpha, \gamma, \varepsilon, \lambda$ | 学习率/折扣/探索率/λ 折中 |
| $G_t, G_{t:t+n}$ | 回报 / n 步回报 |


In [ ]:
# appendix_solve.py —— 附加: 策略迭代的"价值单调性"验证(改进定理数值版)
import numpy as np

SIZE, GAMMA = 4, 0.9
N = SIZE * SIZE
MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c):
    return 0 <= r < SIZE and 0 <= c < SIZE

def trans(s, a):
    if s == N - 1:
        return [(1.0, s, 0.0)]
    r, c = divmod(s, SIZE)
    dr, dc = MOVE[a]
    nr, nc = r + dr, c + dc
    if not in_bounds(nr, nc):
        return [(1.0, s, -1.0)]
    s2 = nr * SIZE + nc
    return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]

def eval_pi(pi):
    V = np.zeros(N)
    for _ in range(500):
        Vn = np.zeros(N)
        for s in range(N):
            Vn[s] = sum(pi[s, a] * p * (r + GAMMA * V[s2])
                        for a in range(4) for p, s2, r in trans(s, a))
        V = Vn
    return V

pi = np.full((N, 4), 0.25)
values = []
for k in range(6):
    V = eval_pi(pi)
    values.append(V[0])
    pi_new = np.zeros((N, 4))
    for s in range(N):
        q = [sum(p * (r + GAMMA * V[s2]) for p, s2, r in trans(s, a)) for a in range(4)]
        pi_new[s, int(np.argmax(q))] = 1.0
    if np.array_equal(pi, pi_new):
        break
    pi = pi_new

print('策略迭代 起点价值逐轮(单调不减):')
for k, v in enumerate(values):
    print(f'  第{k}轮: {v:.3f}')
print('单调性:', '满足' if all(values[i] <= values[i+1] + 1e-9 for i in range(len(values)-1)) else '不满足')
print('→ 改进定理的数值见证: 价值单调上升直到最优')


## 25. 本章手写模板（🔴 面试背诵版）

```python
# 策略评估(同步)
V = np.zeros(N)
for _ in range(max_iter):
    Vn = np.zeros(N)
    for s in range(N):
        Vn[s] = sum(pi[s,a]*p*(r + γ*V[s2]) for a in range(4) for p,s2,r in trans(s,a))
    if np.max(np.abs(Vn-V)) < tol: break
    V = Vn

# 价值迭代(max 版)
V = np.zeros(N)
for _ in range(max_iter):
    Vn = np.zeros(N)
    for s in range(N):
        Vn[s] = max(sum(p*(r+γ*V[s2]) for p,s2,r in trans(s,a)) for a in range(4))
    if np.max(np.abs(Vn-V)) < tol: break
    V = Vn
```

> 两段代码差异只有 `sum(pi[...])` → `max(...)`——面试官改一个词，你要马上反应过来。


## 26. 易错点终版（🔴）

1. 策略评估用**加权平均**、价值迭代用 **max**——被问更新式别混
2. 收敛判据用**最大绝对差**（无穷范数），不是均值差
3. 终点要处理**吸收态**（否则价值会来回跳）
4. `in-place` 更新更快但**严格收敛需顺序扫描**
5. γ→1 时收敛极慢（误差×γ），大 γ 配更多轮
6. 随机转移的 P 是**期望求和**——不要把随机动作当成转移概率


## 27. 考前 3 分钟速背（🔴 DP 篇）

```text
三条更新式:  Σπ(评估) / Σπ+argmax(策略迭代) / max(值迭代)
三个前提:    有模型 / 表格状态 / 静态环境
两个端点:    同步=Jacobi 慢, in-place=Gauss-Seidel 快
一个统一:    一切 RL = GPI(评估+改进)
一个联系:    学模型 → Model-Based RL (Dyna/MuZero)
```


## 28. 常见追问组合演练（🔴 面试对练）

- **A**: "策略评估和值迭代的差别？" → 更新式+用途（各 30 字）
- **B**: "那策略迭代呢？" → 评估到收敛+贪心改进，价值单调升
- **C**: "三个都要模型吗？" → 是；无模型用 MC/TD（下一篇）
- **D**: "大规模怎么办？" → 函数近似 + GPI → DQN/PPO

> 按"定义→对比→举例→延伸"四步答，别停在第一问。


## 29. 模块衔接（🔴 为什么下一篇是 MC/TD）

DP 需要 $P$/$R$——但真实环境往往**给不出**模型。下一篇把"期望求和"换成"采样平均"，一切照样能算——这就是 MC/TD。

> 记忆桥：DP 用代码里的 `trans(s,a)`，MC/TD 用 `step(s,a)`（采样版）。


## 30. DP 变种谱系：同步 / 异步 / 就地 / 优先扫描（🔴 推导关系）

### 谱系树

```text
策略评估(全扫描) ─→ 就地(in-place)更新  ─→ 按优先级顺序更新
                          │
策略迭代 = 评估(收敛) + 贪心改进
值迭代   = 评估(1步)  + 贪心改进          ← 异步版本 = Gauss-Seidel
GPI      = 任意交替评估/改进的统称
```

### 收敛性推导（背一句）

- 同步更新 = Jacobi 迭代：$V_{k+1} = T V_k$，每步 $\\|V_{k+1}-V^*\\|_\\infty \\le \\gamma\\|V_k - V^*\\|_\\infty$
- 异步/就地 = Gauss-Seidel：每个状态用**最新**估计 → 收敛更快且仍收敛（在适当的轮换序下）

> **面试推导点**："就地更新为什么正确？" → 因为贝尔曼算子单调且收缩，用新值替代旧值只会让更新后的状态'更接近'不动点，逐状态处理不破坏收缩性。


## 31. 策略改进定理逐步推导（🔴 面试手推模板）

### 前提

策略 $\pi$ 的 Q 值 $Q^{\pi}(s,a)$，若**逐状态贪心改进**：$\pi'(s) = \arg\max_a Q^{\pi}(s,a)$

### 三步推导（背）

```text
① 单步替换: 贪心保证
   Q^π(s, π'(s)) = max_a Q^π(s,a) ≥ V^π(s)         对每个 s 都成立

② 逐步替换(展开 V^{π'})，把每一步的动作换成 π':
   V^{π'}(s) = E_{π'}[ R + γV^{π'} | s ]
   ≥ E_{π'}[ R + γV^π | s ]        (归纳: 已证 V^{π'} ≥ V^π 在 s' 成立)
   ≥ ...  把 s' 处的 π' 也换成 π ……
   ≥ V^π(s)                        ∎ 价值单调不减
```

### 推论（背）

值迭代/策略迭代里"评估→贪心改进"每一步价值都不降（数值上见过：01 篇 §24 appendix 已验证单调）。
> **口语版**："只要每个状态都换成'不比原来差的动作'，整条策略就整体变好——这就是 GPI 的引擎。"


In [ ]:
# fig_dp_converge.py —— 过程图①: 值迭代误差收敛(不同 γ, log 尺度)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

def vi_errors(gamma, size=4, iters=60):
    N = size * size
    MOVE = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    def trans(s, a):
        if s == N - 1:
            return [(1.0, s, 0.0)]
        r, c = divmod(s, size)
        dr, dc = MOVE[a]
        nr, nc = r + dr, c + dc
        if not (0 <= nr < size and 0 <= nc < size):
            return [(1.0, s, -1.0)]
        s2 = nr * size + nc
        return [(1.0, N - 1, 10.0)] if s2 == N - 1 else [(1.0, s2, -1.0)]
    V = np.zeros(N)
    errs = []
    for _ in range(iters):
        Vn = np.zeros(N)
        for s in range(N):
            Vn[s] = max(sum(p * (r + gamma * V[s2]) for p, s2, r in trans(s, a)) for a in range(4))
        err = np.max(np.abs(Vn - V))
        errs.append(err)
        V = Vn
    return np.array(errs)

fig, ax = plt.subplots(figsize=(7.5, 4.2), dpi=150)
for gamma in [0.5, 0.8, 0.9, 0.95]:
    e = vi_errors(gamma) + 1e-12
    ax.semilogy(e, 'o-', ms=3, label=f'γ={gamma}')
ax.axhline(1e-3, color='red', ls='--', lw=1)
ax.text(1, 1e-3, 'tol=1e-3', color='red', va='bottom')
ax.set_xlabel('迭代轮数 k'); ax.set_ylabel('max|V_{k+1}-V_k| (log)')
ax.set_title('值迭代收敛: 误差按 γ^k 收缩(斜率=log γ)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
out = os.path.join('autumn-recruit-algo', '07-强化学习', '教学', 'images')
if os.path.basename(os.getcwd()) == '教学':
    out = os.path.join(os.getcwd(), 'images')
os.makedirs(out, exist_ok=True)
fig.savefig(os.path.join(out, 'dp_convergence.png'), dpi=150)
print('已保存 images/dp_convergence.png')
plt.close(fig)
plt.show()


In [ ]:
# fig_dp_compare.py —— 过程图②: 策略迭代 vs 值迭代"评估工作量"对比
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

# 教学示意数据: 每轮"底层扫描量"(评估扫描 × 动作) —— 曲线下面积近似总成本
pi_rounds = np.array([500, 2, 500, 2, 500, 2, 2])     # 策略迭代: 评估很长, 改进很短, 少轮
vi_rounds = np.full(24, 30.0)                          # 值迭代: 每轮都轻, 但轮多

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6), dpi=150)
axes[0].bar(range(len(pi_rounds)), pi_rounds, color='steelblue', width=0.6)
axes[0].set_title('策略迭代（少数几轮，每轮评估扫满）', fontsize=10)
axes[0].set_xlabel('轮次'); axes[0].set_ylabel('单轮扫描量')
axes[1].bar(range(len(vi_rounds)), vi_rounds, color='seagreen', width=0.6)
axes[1].set_title('值迭代（很多轮，每轮只扫一遍）', fontsize=10)
axes[1].set_xlabel('轮次'); axes[1].set_ylabel('单轮扫描量')
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
out = os.path.join('autumn-recruit-algo', '07-强化学习', '教学', 'images')
if os.path.basename(os.getcwd()) == '教学':
    out = os.path.join(os.getcwd(), 'images')
os.makedirs(out, exist_ok=True)
fig.savefig(os.path.join(out, 'dp_iters_compare.png'), dpi=150)
print('已保存 images/dp_iters_compare.png')
plt.close(fig)
plt.show()


## 📸 本章推导配图（教学/images/）

![值迭代误差按 γ^k 收缩（log 斜率 = log γ）](images/dp_convergence.png)

> 值迭代误差按 γ^k 收缩（log 斜率 = log γ）

![策略迭代 vs 值迭代的评估成本结构对比](images/dp_iters_compare.png)

> 策略迭代 vs 值迭代的评估成本结构对比
